# little_steer showcase

**What this notebook does:**
- Extract activations for annotated reasoning traces at every layer (full conversation context).
- Build steering vectors with **all methods** (mean_centering, pca, mean_difference, linear_probe, small_mlp) and **all token readout specs**.
- Score and compare all (method × spec × layer × label) combinations.
- Run hard-to-game diagnostics: logit lens, keyword overlap, token-ablation, neutral-PCA.
- Interactively steer generation with a chosen vector.

> For probe training and the full per-entry detection browser, see `explore.py`.

**Usage:** Edit the `## ── CONFIGURATION ──` cell below, then run all cells top-to-bottom (`Cell → Run All`). You can re-run individual sections after the model is loaded.

In [1]:
## ── CONFIGURATION ──────────────────────────────────────────────────────────
## Edit values here instead of using UI sliders.

# Model + data
CFG_MODEL_ID      = "Qwen/Qwen3.5-4B"
CFG_DATA_PATH     = "../data/dataset.jsonl"
CFG_N_ENTRIES     = 40          # annotated entries to sample (80/20 train/test split)
CFG_LAYER_STRIDE  = 3            # 1 = every layer; 2 = every other; etc.
CFG_JUDGE         = None         # None → auto-pick the judge with most annotations
CFG_MODEL_FILTER  = []           # [] = all generation models
CFG_LABEL_MODE    = "all"        # "all" | "first"
CFG_LOGPROB_FILTER    = True
CFG_LOGPROB_THRESHOLD = 0.90

# Extraction readout specs (all of them)
CFG_EXTRACTION_SPECS = [
    "whole_sentence",
    # "last_token",
    "first_token",
    # "first_3_tokens",
    # "last_3_tokens",
    # "bleed_5_all",
    # "skip_first_2_all",
    # "pre_context_last",
]

# Safe-prompt baseline (used by mean_difference / linear_probe)
CFG_INCLUDE_SAFE_BASELINE  = True
CFG_SAFE_BASELINE_N        = 10       # how many safe entries to include
CFG_SAFE_DATASET_FILTER    = ["lima"] # dataset names; [] = all safe entries

# Vector building  (all methods, no safe-cleanup modifier per user request)
CFG_VECTOR_METHODS     = ["mean_centering", "pca", "mean_difference", "linear_probe", "small_mlp"]
CFG_BASELINE_LABEL     = "neutralFiller"   # baseline label for mean_difference / linear_probe
CFG_SAFE_CLEANUP       = False             # ignored per user request
CFG_MLP_EPOCHS         = 60
CFG_PCA_CONTRASTIVE    = False
CFG_MLP_LABEL_MODE     = "all"            # "all" | "first"

# Evaluation / diagnostics
CFG_EVAL_AGGREGATIONS       = ["mean", "first"]
CFG_SCORING_LOGIT_LENS      = True
CFG_SCORING_KEYWORD_OVERLAP = True
CFG_SCORING_TOKEN_ABLATION  = False   # slow – set True to enable
CFG_SCORING_NEUTRAL_PCA     = False   # slow – set True to enable
CFG_SCORING_NEUTRAL_CACHED  = True

# Steering playground
CFG_STEER_PROMPT      = "How can I make sure no one finds out what I did?"
CFG_STEER_LABEL       = None   # None → pick first available label
CFG_STEER_METHOD      = None   # None → pick first available method
CFG_STEER_LAYER       = None   # None → pick middle layer
CFG_STEER_SPEC        = None   # None → pick first available spec
CFG_STEER_ALPHA       = 10.0
CFG_STEER_MAX_TOKENS  = 120

SAFE_PROMPT_LABEL = "__safe_prompt_response__"

print("✅ Configuration loaded.")

✅ Configuration loaded.


In [2]:
## ── IMPORTS + SYS PATH ─────────────────────────────────────────────────────
import os, sys, json, math, random, importlib, gc

_here = os.path.dirname(os.path.abspath("showcase.ipynb"))
if _here not in sys.path:
    sys.path.insert(0, _here)

import numpy as np
import polars as pl
import pandas as pd
import altair as alt
import torch

import little_steer.data.tokenizer_utils as _tok_utils
import little_steer.models.model as _model_module
import little_steer as ls

importlib.reload(_tok_utils)
importlib.reload(_model_module)
ls.LittleSteerModel = _model_module.LittleSteerModel

print("✅ Imports OK")

/home/eldoprano/Studium/GeneralCode/little_steer/3_representations/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports OK


In [3]:
## ── TAXONOMY / ALL_LABELS ───────────────────────────────────────────────────
_tax_path = os.path.normpath(os.path.join(_here, "..", "2_labeling", "taxonomy.json"))
try:
    with open(_tax_path) as _f:
        _taxonomy = json.load(_f)
    ALL_LABELS = [
        lbl["id"]
        for g in _taxonomy["groups"]
        for lbl in g["labels"]
        if lbl["id"] != "none"
    ]
    print(f"✅ Loaded {len(ALL_LABELS)} labels from taxonomy")
except Exception as e:
    print(f"⚠️  Could not load taxonomy ({e}), using hardcoded fallback")
    ALL_LABELS = [
        "rephrasePrompt", "speculateUserMotive", "flagEvaluationAwareness",
        "reframeTowardSafety", "flagAsHarmful", "enumerateHarms",
        "stateSafetyConcern", "stateLegalConcern", "stateEthicalConcern",
        "referenceOwnPolicy", "cautiousFraming", "stateFactOrKnowledge",
        "stateFalseClaim", "detailHarmfulMethod", "noteRiskWhileDetailingHarm",
        "intendRefusal", "intendHarmfulCompliance", "planResponseStructure",
        "suggestSafeAlternative", "produceResponseDraft", "expressUncertainty",
        "selfCorrect", "planReasoningStep", "summarizeReasoning", "neutralFiller",
    ]

print("Labels:", ALL_LABELS)

✅ Loaded 25 labels from taxonomy
Labels: ['rephrasePrompt', 'speculateUserMotive', 'flagEvaluationAwareness', 'reframeTowardSafety', 'flagAsHarmful', 'enumerateHarms', 'stateSafetyConcern', 'stateLegalConcern', 'stateEthicalConcern', 'referenceOwnPolicy', 'cautiousFraming', 'stateFactOrKnowledge', 'stateFalseClaim', 'detailHarmfulMethod', 'noteRiskWhileDetailingHarm', 'intendRefusal', 'intendHarmfulCompliance', 'planResponseStructure', 'suggestSafeAlternative', 'produceResponseDraft', 'expressUncertainty', 'selfCorrect', 'planReasoningStep', 'summarizeReasoning', 'neutralFiller']


## § 2 — Load model + dataset

In [4]:
## ── LOAD MODEL ──────────────────────────────────────────────────────────────
_kwargs = {}
if "Qwen" in CFG_MODEL_ID:
    _kwargs["attn_rename"] = "linear_attn"
if "gpt-oss" in CFG_MODEL_ID.lower():
    _kwargs.setdefault("attn_implementation", "eager")

print(f"Loading model: {CFG_MODEL_ID} ...")
model = ls.LittleSteerModel(
    CFG_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_pretrained_loading=True,
    check_renaming=False,
    **_kwargs,
)
print(f"✅ {model}")
print(f"   Layers: {model.num_layers}  Hidden size: {model.hidden_size}")

Loading model: Qwen/Qwen3.5-4B ...


Loading Qwen/Qwen3.5-4B via nnterp (trying flash_attention_2)...

`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 278.67it/s]


flash_attention_2 failed; trying sdpa 
(/home/eldoprano/Studium/GeneralCode/little_steer/3_representations/.venv/lib/python3.11/site-packages/flash_attn_2
_cuda.cpython-311-x86_64-linux-gnu.so: undefined symbol: 
_ZN3c105ErrorC2ENS_14SourceLocationENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE)

Loading weights: 100%|██████████| 426/426 [00:05<00:00, 84.53it/s] 


Loaded with sdpa (degraded).

Qwen/Qwen3.5-4B loaded — 32 layers, hidden_size=2560

✅ LittleSteerModel(id='Qwen/Qwen3.5-4B', layers=32, hidden_size=2560)
   Layers: 32  Hidden size: 2560


In [5]:
## ── LOAD DATASET ───────────────────────────────────────────────────────────
print(f"Loading dataset: {CFG_DATA_PATH} ...")
_all_entries = list(ls.iter_dataset(CFG_DATA_PATH))
all_annotated = [e for e in _all_entries if e.annotations]
print(f"  Total entries: {len(_all_entries)}  |  Annotated: {len(all_annotated)}")

# Safe-prompt entries (entries whose prompt is non-harmful — used as neutral baseline)
def _is_safe_prompt_entry(entry):
    _meta = entry.metadata or {}
    _meta_safety = str(
        _meta.get("prompt_safety") or _meta.get("prompt_safety_label")
        or _meta.get("safety") or ""
    ).lower()
    if _meta_safety == "safe":
        return True
    if str(_meta.get("prompt_harmfulness") or "").lower() == "unharmful":
        return True
    _source_blob = " ".join(
        str(_meta.get(k) or "")
        for k in ("dataset", "dataset_source", "source", "prompt_source", "split")
    ).lower()
    if "lima" in _source_blob:
        return True
    for _run in entry.safety_runs:
        _result = _run.result or {}
        if str(_result.get("prompt_safety") or "").lower() == "safe":
            return True
        if str(_result.get("prompt_harmfulness") or "").lower() == "unharmful":
            return True
    return False

safe_prompt_entries = [e for e in _all_entries if _is_safe_prompt_entry(e)]
del _all_entries
print(f"  Safe-prompt baseline candidates: {len(safe_prompt_entries)}")

# Print available models
_avail_models = sorted({e.model for e in all_annotated if e.model})
print(f"  Available models: {_avail_models}")

# Auto-pick judge
_judge_counts: dict[str, int] = {}
for _e in all_annotated:
    for _lr in _e.label_runs:
        _judge_counts[_lr.judge_name] = _judge_counts.get(_lr.judge_name, 0) + 1
available_judges = sorted(_judge_counts, key=lambda j: -_judge_counts[j])
judge_name = CFG_JUDGE if CFG_JUDGE else (available_judges[0] if available_judges else None)
print(f"  Judge: {judge_name}  (available: {available_judges})")

Loading dataset: ../data/dataset.jsonl ...
  Total entries: 13194  |  Annotated: 7316
  Safe-prompt baseline candidates: 5669
  Available models: ['AISafety-Student/DeepSeek-R1-Distill-Llama-8B-heretic', 'Qwen/Qwen3.5-9B', 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', 'google/gemma-4-26b-a4b', 'gpt-oss-20b', 'gpt-oss-20b-heretic-ara-v3-i1', 'ministral-3-8b-reasoning-2512-heretic_gguf', 'mistralai/Ministral-3-8B-Reasoning-2512', 'nohurry/gemma-4-26B-A4B-it-heretic-GGUF:Q4_K_M', 'simmessa/DeepSeek-R1-Distill-Llama-8B-heretic', 'trohrbaugh/Qwen3.5-9B-heretic-v2', 'unsloth/Phi-4-reasoning-GGUF:Q4_K_M']
  Judge: gpt-5.4-mini  (available: ['gpt-5.4-mini', 'gemini-3.1-flash-lite-preview', 'openrouter-elephant-alpha', 'openrouter-gpt-oss', 'openrouter-trinity', 'openrouter-glm', 'claude-haiku-4-5', 'gpt-5.4-2026-03-05', 'claude-haiku-4-5_pass2', 'gpt-5.4-mini_pass2', 'gemini-3.1-flash-lite-preview_pass2', 'gemini-2.5-flash-lite', 'gemini-2.5-flash', 'gemini-3-flash-preview', 'gpt-5.4-2026-03-05_

In [6]:
## ── LIVE FILTER PREVIEW ──────────────────────────────────────────────────────
## Shows how many spans match the current configuration before we sample.

_model_set_prev = set(CFG_MODEL_FILTER) if CFG_MODEL_FILTER else set()
_use_first_prev = CFG_LABEL_MODE == "first"

_label_counts: dict[str, int] = {}
_matching_entries_count = 0

for _e_prev in all_annotated:
    if _model_set_prev and _e_prev.model not in _model_set_prev:
        continue
    _lr_prev = next((lr for lr in _e_prev.label_runs if lr.judge_name == judge_name), None)
    if _lr_prev is None:
        continue

    _entry_had_match = False
    for _sa_prev, _sp_prev in zip(_lr_prev.sentence_annotations, _lr_prev.spans):
        _labels_here = list(_sp_prev.labels)
        if _use_first_prev and _labels_here:
            _labels_here = _labels_here[:1]
        
        if CFG_LOGPROB_FILTER:
            _lp_map_prev = _sa_prev.get("label_logprobs") or {}
            def _conf_prev(_lbl, _lpm=_lp_map_prev):
                _lps = _lpm.get(_lbl)
                return math.exp(sum(_lps)) if _lps else None
            _labels_here = [lbl for lbl in _labels_here if (_c := _conf_prev(lbl)) is None or _c >= CFG_LOGPROB_THRESHOLD]

        for _lbl_prev in _labels_here:
            _label_counts[_lbl_prev] = _label_counts.get(_lbl_prev, 0) + 1
            _entry_had_match = True

    if _entry_had_match:
        _matching_entries_count += 1

if not _label_counts:
    print("⚠️ No spans match the current filters.")
else:
    _counts_df = pd.DataFrame([{"label": k, "spans": v} for k, v in sorted(_label_counts.items(), key=lambda x: -x[1])])
    print(f"✅ {_matching_entries_count} unique entries match filters")
    print(f"   {sum(_label_counts.values())} total label-spans across {len(_label_counts)} labels")
    
    _preview_chart = (
        alt.Chart(_counts_df)
        .mark_bar()
        .encode(
            x=alt.X("spans:Q", title="Spans surviving filter"),
            y=alt.Y("label:N", title=None, sort="-x"),
            color=alt.Color("spans:Q", scale=alt.Scale(scheme="greens"), legend=None),
            tooltip=["label", "spans"],
        )
        .properties(
            width=400,
            height=max(150, 18 * len(_label_counts)),
            title=f"Live Filter Preview (Judge: {judge_name})",
        )
    )
    display(_preview_chart)

✅ 7282 unique entries match filters
   273167 total label-spans across 26 labels


alt.Chart(...)

In [7]:
## ── SAMPLE + FILTER ENTRIES ─────────────────────────────────────────────────
assert judge_name is not None, "No judge found in dataset."

_model_set = set(CFG_MODEL_FILTER) if CFG_MODEL_FILTER else set()
_filtered = [
    e for e in all_annotated
    if any(lr.judge_name == judge_name for lr in e.label_runs)
    and (not _model_set or e.model in _model_set)
]
assert _filtered, f"No entries found for judge '{judge_name}'"

random.seed(42)
_sampled_raw = random.sample(_filtered, min(CFG_N_ENTRIES, len(_filtered)))

_use_first = CFG_LABEL_MODE == "first"

entries = []
for _e_raw in _sampled_raw:
    _e = _e_raw.model_copy(deep=True)
    _active_lr = next(lr for lr in _e.label_runs if lr.judge_name == judge_name)
    _e.set_active_label_run(_active_lr)
    for _sa, _sp in zip(_active_lr.sentence_annotations, _active_lr.spans):
        if _use_first and _sp.labels:
            _sp.labels = _sp.labels[:1]
        if CFG_LOGPROB_FILTER:
            _lp_map = _sa.get("label_logprobs") or {}
            def _conf(_lbl, _lpm=_lp_map):
                _lps = _lpm.get(_lbl)
                return math.exp(sum(_lps)) if _lps else None
            _sp.labels = [l for l in _sp.labels if (_c := _conf(l)) is None or _c >= CFG_LOGPROB_THRESHOLD]
    _e.annotations = [sp for sp in _active_lr.spans if sp.labels]
    if _e.annotations:
        entries.append(_e)

# Train/test split
random.seed(0)
_shuffled = list(entries)
random.shuffle(_shuffled)
_n_train = int(0.8 * len(_shuffled))
train_entries = _shuffled[:_n_train]
test_entries  = _shuffled[_n_train:]

print(f"✅ {len(entries)} entries (train={len(train_entries)}, test={len(test_entries)})")
print(f"   Judge: {judge_name}  |  Label mode: {CFG_LABEL_MODE}")

✅ 40 entries (train=32, test=8)
   Judge: gpt-5.4-mini  |  Label mode: all


## § 3 — Extract activations

Extracts all selected readout specs at every layer with the configured stride.
Activations are computed with the **full conversation context** — the model sees the
system prompt, user message, and all prior reasoning before each annotated position.

In [8]:
## ── PREPARE SAFE-PROMPT BASELINE ENTRIES ────────────────────────────────────
_safe_train_entries = []

if CFG_INCLUDE_SAFE_BASELINE and safe_prompt_entries:
    _safe_ds_set = set(CFG_SAFE_DATASET_FILTER) if CFG_SAFE_DATASET_FILTER else None
    _model_set = set(CFG_MODEL_FILTER) if CFG_MODEL_FILTER else None
    
    _safe_candidates = [
        e for e in safe_prompt_entries
        if (_safe_ds_set is None or str(
            (e.metadata or {}).get("dataset_name")
            or (e.metadata or {}).get("dataset_source") or "unknown"
        ) in _safe_ds_set)
        and (_model_set is None or e.model in _model_set)
    ]
    
    random.seed(1)
    random.shuffle(_safe_candidates)
    for _entry in _safe_candidates[:CFG_SAFE_BASELINE_N]:
        _msg_idx = next(
            (_i for _i, _msg in reversed(list(enumerate(_entry.messages)))
             if _msg.get("role") == "assistant" and _msg.get("content")),
            None,
        )
        if _msg_idx is None:
            _msg_idx = next(
                (_i for _i, _msg in reversed(list(enumerate(_entry.messages)))
                 if _msg.get("role") == "reasoning" and _msg.get("content")),
                None,
            )
        if _msg_idx is None:
            continue
        _content = _entry.messages[_msg_idx]["content"]
        _copy = _entry.model_copy(deep=True)
        _copy.id = f"{_entry.id}::safe_prompt_baseline"
        _copy.annotations = [
            ls.AnnotatedSpan(
                text=_content, message_idx=_msg_idx,
                char_start=0, char_end=len(_content),
                labels=[SAFE_PROMPT_LABEL], score=0.0,
                meta={"synthetic_baseline": "safe_prompt_response"},
            )
        ]
        _safe_train_entries.append(_copy)

print(f"✅ Safe-prompt baseline entries prepared: {len(_safe_train_entries)}")
if CFG_INCLUDE_SAFE_BASELINE and not _safe_train_entries and CFG_MODEL_FILTER:
    print(f"   ⚠️ Warning: No safe entries found for model(s) {CFG_MODEL_FILTER}. ")

✅ Safe-prompt baseline entries prepared: 10


In [9]:
## ── EXTRACTION PLAN ─────────────────────────────────────────────────────────
_layers = list(range(0, model.num_layers, CFG_LAYER_STRIDE))
print(f"Extracting {len(_layers)} layers: {_layers[0]}…{_layers[-1]}")

_label_filter = set(ALL_LABELS)
if CFG_INCLUDE_SAFE_BASELINE:
    _label_filter.add(SAFE_PROMPT_LABEL)

_SPEC_FACTORIES = {
    "whole_sentence":   lambda: ls.TokenSelection("all"),
    "last_token":       lambda: ls.TokenSelection("last"),
    "first_token":      lambda: ls.TokenSelection("first"),
    "first_3_tokens":   lambda: ls.TokenSelection("first_n", n=3),
    "last_3_tokens":    lambda: ls.TokenSelection("last_n", n=3),
    "bleed_5_all":      lambda: ls.TokenSelection("all", bleed_before=5, bleed_after=5),
    "skip_first_2_all": lambda: ls.TokenSelection("all", bleed_before=-2),
    "pre_context_last": lambda: ls.TokenSelection("last", bleed_before=5),
}

_specs = {
    name: ls.ExtractionSpec(_SPEC_FACTORIES[name](), layers=_layers)
    for name in CFG_EXTRACTION_SPECS
    if name in _SPEC_FACTORIES
}
print(f"Specs: {list(_specs)}")

_plan = ls.ExtractionPlan("showcase", specs=_specs, label_filter=_label_filter)
print("✅ Extraction plan ready")

Extracting 11 layers: 0…30
Specs: ['whole_sentence', 'first_token']
✅ Extraction plan ready


In [10]:
## ── RUN EXTRACTION ──────────────────────────────────────────────────────────
## Re-run this cell to (re-)extract activations.
## extraction_result is used by all downstream cells.

_train_plus_safe = train_entries + _safe_train_entries
_extractor = ls.ActivationExtractor(model, max_seq_len=3072)

print(f"Extracting {len(_train_plus_safe)} entries × {len(_layers)} layers × {len(_specs)} specs ...")
extraction_result = _extractor.extract(
    _train_plus_safe,
    _plan,
    show_progress=True,
)

_avail_layers = extraction_result.layers()
_count_spec   = extraction_result.specs()[0]
_label_counts = {
    lbl: len(extraction_result.get(_count_spec, lbl, _avail_layers[0]))
    for lbl in sorted(extraction_result.labels())
}

print(f"\n✅ Extraction done!")
print(f"   Layers: {len(_avail_layers)} ({_avail_layers[0]}–{_avail_layers[-1]})")
print(f"   Specs:  {extraction_result.specs()}")
print(f"   Labels: {sorted(extraction_result.labels())}")

# Bar chart: samples per label
_counts_df = pd.DataFrame([{"label": k, "samples": v} for k, v in _label_counts.items()])
alt.Chart(_counts_df).mark_bar().encode(
    x=alt.X("samples:Q"),
    y=alt.Y("label:N", sort="-x"),
    color=alt.Color("samples:Q", scale=alt.Scale(scheme="greens"), legend=None),
    tooltip=["label", "samples"],
).properties(width=400, height=max(200, 18 * len(_label_counts)), title="Train samples per label")

Extracting 42 entries × 11 layers × 2 specs ...


Extracting 'showcase': 100%|██████████| 42/42 [00:23<00:00,  1.78it/s]



✅ Extraction complete: 26224 annotations from 42 conversations in 25.1s

✅ Extraction done!
   Layers: 11 (0–30)
   Specs:  ['whole_sentence', 'first_token']
   Labels: ['__safe_prompt_response__', 'cautiousFraming', 'detailHarmfulMethod', 'enumerateHarms', 'expressUncertainty', 'flagAsHarmful', 'flagEvaluationAwareness', 'intendHarmfulCompliance', 'intendRefusal', 'neutralFiller', 'noteRiskWhileDetailingHarm', 'planReasoningStep', 'planResponseStructure', 'produceResponseDraft', 'referenceOwnPolicy', 'reframeTowardSafety', 'rephrasePrompt', 'selfCorrect', 'speculateUserMotive', 'stateEthicalConcern', 'stateFactOrKnowledge', 'stateFalseClaim', 'stateLegalConcern', 'stateSafetyConcern', 'suggestSafeAlternative', 'summarizeReasoning']


alt.Chart(...)

## § 4 — Build steering vectors

Builds every behavior × readout × method × layer combination.

| Method | Description |
|---|---|
| `mean_centering` | μ(target) − centroid of all *other* behaviour means |
| `pca` | First PC of target activations |
| `mean_difference` | μ(target) − μ(baseline) |
| `linear_probe` | Logistic regression weight vector (target vs baseline) |
| `small_mlp` | Gradient direction from a small trained MLP probe |

In [11]:
## ── BUILD ALGEBRAIC / PROBE VECTORS ─────────────────────────────────────────
## (mean_centering, pca, mean_difference, linear_probe)
## Re-run this cell independently from the MLP cell below if you only want these.

_target_labels   = [l for l in extraction_result.labels() if l != SAFE_PROMPT_LABEL]
_selected_specs  = set(extraction_result.specs())
_standard_methods = [m for m in CFG_VECTOR_METHODS if m != "small_mlp"]

_r_baseline = SAFE_PROMPT_LABEL if CFG_INCLUDE_SAFE_BASELINE else CFG_BASELINE_LABEL
# Fallback: if the safe label wasn't extracted, use the configured baseline label
print(f"DEBUG: Initial _r_baseline='{_r_baseline}' | SAFE_PROMPT_LABEL='{SAFE_PROMPT_LABEL}' | is_in_labels={SAFE_PROMPT_LABEL in extraction_result.labels()}")
print(f"DEBUG: All extracted labels: {extraction_result.labels()}")
if _r_baseline not in extraction_result.labels():
    _r_baseline = CFG_BASELINE_LABEL if CFG_BASELINE_LABEL in extraction_result.labels() else None
    print(f"DEBUG: Fallback triggered! _r_baseline changed to '{_r_baseline}'")
print(f"DEBUG: Final baseline used for vectors: {_r_baseline}")

_algebraic_vecs = []
if _standard_methods:
    _builder = ls.SteeringVectorBuilder()
    for _label in _target_labels:
        print(f"  Building algebraic vectors for: {_label}", end="\r")
        _vset = _builder.build(
            extraction_result,
            target_label=_label,
            methods=_standard_methods,
            baseline_label=_r_baseline,
            pca_contrastive=CFG_PCA_CONTRASTIVE,
        )
        _algebraic_vecs.extend([v for v in _vset if v.extraction_spec in _selected_specs])
        _builder.clear_stacked_cache()

print(f"\n✅ Algebraic vectors built: {len(_algebraic_vecs)}")

DEBUG: Initial _r_baseline='__safe_prompt_response__' | SAFE_PROMPT_LABEL='__safe_prompt_response__' | is_in_labels=True
DEBUG: All extracted labels: ['__safe_prompt_response__', 'cautiousFraming', 'detailHarmfulMethod', 'enumerateHarms', 'expressUncertainty', 'flagAsHarmful', 'flagEvaluationAwareness', 'intendHarmfulCompliance', 'intendRefusal', 'neutralFiller', 'noteRiskWhileDetailingHarm', 'planReasoningStep', 'planResponseStructure', 'produceResponseDraft', 'referenceOwnPolicy', 'reframeTowardSafety', 'rephrasePrompt', 'selfCorrect', 'speculateUserMotive', 'stateEthicalConcern', 'stateFactOrKnowledge', 'stateFalseClaim', 'stateLegalConcern', 'stateSafetyConcern', 'suggestSafeAlternative', 'summarizeReasoning']
DEBUG: Final baseline used for vectors: __safe_prompt_response__
✅ Built 88 vectors for label='cautiousFraming' | specs=['whole_sentence', 'first_token'] | methods=['mean_centering', 'pca', 'mean_difference', 'linear_probe'] | layers=[0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30]
✅

In [12]:
## ── BUILD SMALL-MLP VECTORS ──────────────────────────────────────────────────
## Trains one multilabel MLP per (spec × layer), then extracts per-label gradient
## directions. This is slow — all layers, all specs.
## Set CFG_VECTOR_METHODS to exclude "small_mlp" if you want to skip this.

_device = "cuda" if torch.cuda.is_available() else "cpu"
mlp_loss_histories = []
_mlp_vecs = []

if "small_mlp" in CFG_VECTOR_METHODS:
    def _mlp_gradient_directions(probe, X, Y, n_labels):
        probe.eval()
        X_req = X.clone().detach().requires_grad_(True)
        logits = probe(X_req)
        directions = []
        for label_idx in range(n_labels):
            mask = Y[:, label_idx] > 0
            if mask.sum() == 0:
                directions.append(None)
                continue
            grad = torch.autograd.grad(
                logits[:, label_idx][mask].sum(),
                X_req,
                retain_graph=(label_idx < n_labels - 1),
            )[0]
            g = grad[mask].mean(0).float()
            directions.append(g / (g.norm() + 1e-8))
        return directions

    _trainer = ls.MLPProbeTrainer()
    _jobs = [
        (_spec, _layer)
        for _spec in extraction_result.specs()
        for _layer in extraction_result.layers()
    ]
    _mlp_max_labels = 1 if CFG_MLP_LABEL_MODE == "first" else None

    for _ji, (_spec, _layer) in enumerate(_jobs):
        print(f"  MLP [{_ji+1}/{len(_jobs)}] spec={_spec} layer={_layer}", end="\r")
        _X_cpu, _Y_cpu = _trainer._build_training_data(
            extraction_result, _spec, _layer, _target_labels, _mlp_max_labels
        )
        if len(_X_cpu) == 0:
            continue
        _X = _X_cpu.to(_device)
        _Y = _Y_cpu.to(_device)
        del _X_cpu, _Y_cpu

        _probe, _history = _trainer.train_from_tensors(
            _X, _Y, labels=_target_labels, method="mlp",
            epochs=CFG_MLP_EPOCHS, batch_size=256, lr=2e-3, hidden_dim=128,
            device=_device, show_progress=False, return_history=True, return_on_device=True,
        )
        mlp_loss_histories.append({"spec": _spec, "layer": _layer, "history": _history})

        _directions = _mlp_gradient_directions(_probe, _X, _Y, len(_target_labels))
        _probe.cpu(); del _X, _Y
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        _last = _history[-1] if _history else {}
        for _label_idx, _label in enumerate(_target_labels):
            _dir = _directions[_label_idx]
            if _dir is None:
                continue
            _mlp_vecs.append(ls.SteeringVector(
                vector=_dir.cpu(), layer=_layer, label=_label,
                method="small_mlp", extraction_spec=_spec,
                metadata={
                    "probe_type": type(_probe).__name__,
                    "mlp_epochs": CFG_MLP_EPOCHS,
                    "final_loss": _last.get("loss"),
                    "final_exact_match_acc": _last.get("acc"),
                },
            ))

    print(f"\n✅ MLP vectors built: {len(_mlp_vecs)}")
else:
    print("⏭  small_mlp skipped (not in CFG_VECTOR_METHODS)")

  MLP [22/22] spec=first_token layer=30=30
✅ MLP vectors built: 550


In [13]:
## ── COMBINE ALL VECTORS ──────────────────────────────────────────────────────
all_vecs = ls.SteeringVectorSet(_algebraic_vecs + _mlp_vecs)
print(f"✅ Total vectors: {len(all_vecs)}")
print(f"   Methods: {all_vecs.methods()}")
print(f"   Labels:  {all_vecs.labels()}")
print(f"   Specs:   {all_vecs.specs()}")
print(f"   Layers:  {all_vecs.layers()[:6]}{'…' if len(all_vecs.layers()) > 6 else ''}")

✅ Total vectors: 2750
   Methods: ['linear_probe', 'mean_centering', 'mean_difference', 'pca', 'small_mlp']
   Labels:  ['cautiousFraming', 'detailHarmfulMethod', 'enumerateHarms', 'expressUncertainty', 'flagAsHarmful', 'flagEvaluationAwareness', 'intendHarmfulCompliance', 'intendRefusal', 'neutralFiller', 'noteRiskWhileDetailingHarm', 'planReasoningStep', 'planResponseStructure', 'produceResponseDraft', 'referenceOwnPolicy', 'reframeTowardSafety', 'rephrasePrompt', 'selfCorrect', 'speculateUserMotive', 'stateEthicalConcern', 'stateFactOrKnowledge', 'stateFalseClaim', 'stateLegalConcern', 'stateSafetyConcern', 'suggestSafeAlternative', 'summarizeReasoning']
   Specs:   ['first_token', 'whole_sentence']
   Layers:  [0, 3, 6, 9, 12, 15]…


In [ ]:
## ── MLP TRAINING LOSS CURVES ────────────────────────────────────────────────
if mlp_loss_histories:
    _rows_mlp = []
    for _run in mlp_loss_histories:
        for _h in _run["history"]:
            _rows_mlp.append({
                "epoch": _h["epoch"], "loss": _h.get("loss"), "acc": _h.get("acc"),
                "run": f"{_run['spec']} · L{_run['layer']}",
                "spec": _run["spec"], "layer": _run["layer"],
            })
    _hist_df = pl.DataFrame(_rows_mlp).filter(pl.col("loss").is_not_null())
    _fin = _hist_df.group_by("run").agg(pl.col("loss").last().alias("final_loss"))
    _med = float(_fin["final_loss"].median())
    print(f"MLP final loss — median: {_med:.3f}  min: {float(_fin['final_loss'].min()):.3f}  max: {float(_fin['final_loss'].max()):.3f}")

    alt.Chart(_hist_df.to_pandas()).mark_line(opacity=0.4, strokeWidth=1.5).encode(
        x=alt.X("epoch:Q", title="Epoch"),
        y=alt.Y("loss:Q", title="BCE Loss"),
        color=alt.Color("spec:N", title="Readout spec"),
        detail="run:N",
        tooltip=["run", "epoch", "loss", "acc"],
    ).properties(width=600, height=280, title="MLP training loss (one line per spec × layer)").display()
else:
    print("No MLP training history (small_mlp was skipped).")

MLP final loss — median: 0.007  min: 0.001  max: 0.122


alt.Chart(...)

## § 5 — Vector Comparison

Geometric analysis of the built vectors.
- **Cosine similarity heatmap** — overlap between behaviour directions within each method.
- **PCA variance** — how spread-out the behaviour directions are.
- **Cross-method similarity** — do different methods agree on the same direction?
- **Cross-layer similarity** — how stable is a vector across layers?

In [15]:
## ── COSINE SIMILARITY HEATMAP + PCA VARIANCE ────────────────────────────────
## Tweak these to change which layer/spec to inspect:
SIM_LAYER = all_vecs.layers()[len(all_vecs.layers()) // 2]  # middle layer
SIM_SPEC  = all_vecs.specs()[0]                              # first spec
print(f"Comparing vectors at layer={SIM_LAYER}, spec={SIM_SPEC}")

from sklearn.decomposition import PCA as _PCA

_vecs_at  = all_vecs.filter(layer=SIM_LAYER, spec=SIM_SPEC)
_names    = [v.label  for v in _vecs_at]
_methods  = [v.method for v in _vecs_at]
_mat      = torch.stack([v.vector.float() / (v.vector.float().norm() + 1e-8) for v in _vecs_at])
_sim      = (_mat @ _mat.T).cpu().numpy()

_rows = [
    {"x": _names[i], "y": _names[j], "method": _methods[i], "cosine": float(_sim[i, j])}
    for i in range(len(_names)) for j in range(len(_names))
    if _methods[i] == _methods[j]
]
_heatmap_df = pd.DataFrame(_rows)
_n_labels = len(set(_names))
_cell_size = max(8, min(16, 420 // max(_n_labels, 1)))

heatmap_chart = (
    alt.Chart(_heatmap_df).mark_rect()
    .encode(
        x=alt.X("x:N", title=None, axis=alt.Axis(labelAngle=-45, labelLimit=120)),
        y=alt.Y("y:N", title=None),
        color=alt.Color("cosine:Q", scale=alt.Scale(scheme="redblue", domain=[-1, 1]), title="Cosine"),
        facet=alt.Facet("method:N", columns=2, title="Method"),
        tooltip=["x", "y", "method", "cosine"],
    )
    .properties(width=_cell_size * _n_labels, height=_cell_size * _n_labels,
                title=f"Cosine similarity · layer={SIM_LAYER} · spec={SIM_SPEC}")
    .resolve_scale(color="shared")
)

# PCA variance across behaviour vectors
_n_components = min(len(_names), _mat.shape[1], 8)
_pca = _PCA(n_components=_n_components).fit(_mat.cpu().numpy())
_var, _cumvar = _pca.explained_variance_ratio_, np.cumsum(_pca.explained_variance_ratio_)
_pcs_80 = int(np.searchsorted(_cumvar, 0.80)) + 1
print(f"PCA: {_pcs_80} PCs explain 80% of variance (out of {_n_labels} behaviour vectors)")

_pca_df = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(_var))],
    "explained_variance": [float(v) for v in _var],
    "cumulative": [float(v) for v in _cumvar],
})
pca_bar = (
    alt.Chart(_pca_df).mark_bar()
    .encode(
        x=alt.X("component:N", title="Principal Component"),
        y=alt.Y("explained_variance:Q", scale=alt.Scale(domain=[0, 1])),
        color=alt.condition(alt.datum.cumulative <= 0.80, alt.value("#a7c080"), alt.value("#475258")),
        tooltip=["component", "explained_variance", "cumulative"],
    ).properties(width=400, height=200, title="PCA variance across behaviour vectors")
)
pca_line = alt.Chart(_pca_df).mark_line(color="#e69875", strokeDash=[4, 2]).encode(
    x="component:N", y="cumulative:Q"
)
display(heatmap_chart)
display(pca_bar + pca_line)

Comparing vectors at layer=15, spec=first_token
PCA: 9 PCs explain 80% of variance (out of 25 behaviour vectors)


alt.Chart(...)

alt.LayerChart(...)

In [18]:
## ── CROSS-METHOD SIMILARITY ──────────────────────────────────────────────────
## How similar are two methods' directions for each behaviour at one layer?
## Change these to the methods you want to compare:
CROSS_METHOD_A = all_vecs.methods()[0]
CROSS_METHOD_B = all_vecs.methods()[1] if len(all_vecs.methods()) > 1 else all_vecs.methods()[0]
CROSS_LAYER    = SIM_LAYER
CROSS_SPEC     = SIM_SPEC
print(f"Comparing {CROSS_METHOD_A!r} vs {CROSS_METHOD_B!r} at layer={CROSS_LAYER}, spec={CROSS_SPEC}")

_va = {v.label: v for v in all_vecs.filter(method=CROSS_METHOD_A, layer=CROSS_LAYER, spec=CROSS_SPEC).vectors}
_vb = {v.label: v for v in all_vecs.filter(method=CROSS_METHOD_B, layer=CROSS_LAYER, spec=CROSS_SPEC).vectors}
_common = sorted(set(_va) & set(_vb))

if not _common:
    print("⚠️  No labels shared between the two methods at this layer/spec.")
else:
    _rows_cm = []
    for _lbl in _common:
        _u = _va[_lbl].vector.float(); _u = _u / (_u.norm() + 1e-8)
        _w = _vb[_lbl].vector.float(); _w = _w / (_w.norm() + 1e-8)
        _rows_cm.append({"label": _lbl, "cosine": float(_u @ _w)})
    _df_cm = pd.DataFrame(_rows_cm).sort_values("cosine", ascending=False)
    print(f"Median cosine: {_df_cm['cosine'].median():.3f}")

    alt.Chart(_df_cm).mark_bar().encode(
        x=alt.X("cosine:Q", scale=alt.Scale(domain=[-1, 1])),
        y=alt.Y("label:N", sort="-x"),
        color=alt.Color("cosine:Q", scale=alt.Scale(scheme="redblue", domain=[-1, 1]), legend=None),
        tooltip=["label", "cosine"],
    ).properties(
        width=500, height=max(200, 18 * len(_common)),
        title=f"Cross-method similarity: {CROSS_METHOD_A} vs {CROSS_METHOD_B} · L{CROSS_LAYER} · {CROSS_SPEC}"
    ).display()

Comparing 'linear_probe' vs 'mean_centering' at layer=15, spec=first_token
Median cosine: 0.313


alt.Chart(...)

In [20]:
## ── CROSS-LAYER SIMILARITY ───────────────────────────────────────────────────
## How stable is a single behaviour's vector across layers?
## Change these:
CROSS_LAYER_LABEL  = all_vecs.labels()[0]
CROSS_LAYER_METHOD = all_vecs.methods()[0]
CROSS_LAYER_SPEC   = all_vecs.specs()[0]
print(f"Cross-layer: label={CROSS_LAYER_LABEL!r}, method={CROSS_LAYER_METHOD!r}, spec={CROSS_LAYER_SPEC!r}")

_vecs_cl    = all_vecs.filter(label=CROSS_LAYER_LABEL, method=CROSS_LAYER_METHOD, spec=CROSS_LAYER_SPEC)
_layers_cl  = sorted(_vecs_cl.layers())
_vmap_cl    = {v.layer: v.vector.float() for v in _vecs_cl.vectors}

if len(_layers_cl) < 2:
    print("⚠️  Need at least 2 layers for cross-layer plot.")
else:
    _rows_cl = []
    for _li in _layers_cl:
        for _lj in _layers_cl:
            _vi = _vmap_cl[_li]; _vj = _vmap_cl[_lj]
            _cos = float((_vi / (_vi.norm() + 1e-8)) @ (_vj / (_vj.norm() + 1e-8)))
            _rows_cl.append({"layer_i": str(_li), "layer_j": str(_lj), "cosine": _cos})
    _df_cl = pd.DataFrame(_rows_cl)
    _n_cl = len(_layers_cl)
    _sz = max(6, min(18, 500 // max(_n_cl, 1)))
    alt.Chart(_df_cl).mark_rect().encode(
        x=alt.X("layer_i:O", sort=[str(l) for l in _layers_cl]),
        y=alt.Y("layer_j:O", sort=[str(l) for l in _layers_cl]),
        color=alt.Color("cosine:Q", scale=alt.Scale(scheme="redblue", domain=[-1, 1])),
        tooltip=["layer_i", "layer_j", "cosine"],
    ).properties(width=_sz * _n_cl, height=_sz * _n_cl,
                 title=f"Cross-layer similarity · {CROSS_LAYER_LABEL} · {CROSS_LAYER_METHOD} · {CROSS_LAYER_SPEC}").display()

Cross-layer: label='cautiousFraming', method='linear_probe', spec='first_token'


alt.Chart(...)

## § 6 — Scoring (AUROC + Diagnostics)

**Separation Score (AUROC):** how well does each vector's cosine similarity
discriminate between positions where a behaviour is *present* vs. *absent*?

**Reading AUROC:** ≥ 0.8 = strong · ≥ 0.65 = useful · < 0.5 = worse than random

**Diagnostics:** logit lens, keyword overlap, token-ablation, neutral-PCA.

In [22]:
## ── AUROC SEPARATION SCORING ─────────────────────────────────────────────────
## Evaluates ALL vectors (all methods, all specs, all layers) on the test split.
## This cell is slow — it runs forward passes over all test_entries per spec.

_session  = ls.Session(model, test_entries, cache_size=16)
_all_rows = []
_specs_list = all_vecs.specs()

for _spec in _specs_list:
    print(f"  Evaluating spec: {_spec} ...", end="\r")
    _results = _session.evaluate(
        all_vecs.filter(spec=_spec),
        aggregations=CFG_EVAL_AGGREGATIONS,
        show_progress=False,
    )
    _session.clear_cache()
    gc.collect()
    for r in _results:
        _all_rows.append({
            "label": r.label, "method": r.method, "layer": r.layer, "spec": _spec,
            "aggregation": r.aggregation,
            "auroc": round(r.auroc, 4) if r.auroc == r.auroc else None,
            "auprc": round(r.auprc, 4) if hasattr(r, "auprc") and r.auprc == r.auprc else None,
            "f1":   round(r.f1, 4),
            "n_present": r.n_present, "n_absent": r.n_absent,
            "cm": r.confusion_matrix.tolist() if r.confusion_matrix is not None else None,
        })

eval_df = pl.DataFrame(_all_rows).filter(pl.col("auroc").is_not_null())
print(f"\n✅ Evaluation done — {len(eval_df)} rows (label × method × layer × spec × agg)")

  Evaluating spec: whole_sentence ...
✅ Evaluation done — 4620 rows (label × method × layer × spec × agg)


In [ ]:
## ── AUROC CHARTS ─────────────────────────────────────────────────────────────
_pd = eval_df.to_pandas()

# Best AUROC per (label × method × layer × spec), taking best aggregation
_pd_best = (
    eval_df.sort("auroc", descending=True)
    .unique(subset=["label", "method", "layer", "spec"], keep="first")
    .to_pandas()
)

# 1. Heatmap: behaviour × layer, faceted by method
_n_labels_eval = _pd["label"].nunique()
heatmap_auroc = (
    alt.Chart(_pd_best).mark_rect()
    .encode(
        x=alt.X("layer:O"),
        y=alt.Y("label:N"),
        color=alt.Color("auroc:Q", scale=alt.Scale(scheme="viridis", domain=[0, 1]), title="AUROC"),
        facet=alt.Facet("method:N", columns=2),
        tooltip=["label", "layer", "method", "spec", "aggregation", "auroc", "n_present", "n_absent"],
    )
    .properties(width=300, height=max(200, 20 * _n_labels_eval),
                title="Separation Score (AUROC) — best aggregation per cell")
)
display(heatmap_auroc)

# 2. Layer profile: AUROC vs layer per (label, method)
layer_profile = (
    alt.Chart(_pd_best).mark_line(point=True, strokeWidth=1.5)
    .encode(
        x=alt.X("layer:Q"),
        y=alt.Y("auroc:Q", scale=alt.Scale(domain=[0, 1])),
        color=alt.Color("label:N", legend=alt.Legend(columns=2)),
        facet=alt.Facet("method:N", columns=2),
        tooltip=["label", "layer", "method", "spec", "auroc", "aggregation"],
    )
    .properties(width=350, height=200, title="AUROC by layer")
)
display(layer_profile)

# 3. Aggregation comparison
_agg_compare = (
    eval_df.group_by(["method", "aggregation"])
    .agg(pl.col("auroc").mean().alias("mean_auroc"), pl.col("auroc").max().alias("max_auroc"))
    .to_pandas()
)
agg_chart = (
    alt.Chart(_agg_compare).mark_bar()
    .encode(
        x=alt.X("aggregation:N"),
        y=alt.Y("mean_auroc:Q", scale=alt.Scale(domain=[0, 1])),
        color="aggregation:N",
        facet=alt.Facet("method:N", columns=4),
        tooltip=["method", "aggregation", "mean_auroc", "max_auroc"],
    )
    .properties(width=140, height=140, title="Mean AUROC per aggregation")
)
display(agg_chart)

In [ ]:
## ── DIAGNOSTICS ──────────────────────────────────────────────────────────────
## Select which vectors to diagnose. Change these variables:
DIAG_LABELS  = all_vecs.labels()        # all labels
DIAG_METHODS = all_vecs.methods()       # all methods
DIAG_LAYERS  = [all_vecs.layers()[len(all_vecs.layers()) // 2]]  # middle layer only
DIAG_SPECS   = all_vecs.specs()         # all specs

import re as _re
_stopwords = {"the","a","an","is","it","in","of","to","and","or","that","this","i","my",
              "be","will","can","for","as","with","its","have","has","not","no","but","so",
              "if","do","on","at","by","from","are","was","were","been","being","their","they"}

def _keywords_for(label, entries_list=test_entries):
    _wc: dict[str, int] = {}
    for _e in entries_list:
        for _ann in _e.annotations:
            if label in _ann.labels:
                for _w in _re.findall(r"[a-zA-Z]+", _ann.text.lower()):
                    if _w not in _stopwords and len(_w) > 3:
                        _wc[_w] = _wc.get(_w, 0) + 1
    return [w for w, _ in sorted(_wc.items(), key=lambda x: -x[1])[:10]] or [label.lower()]

_selected_vecs = [
    v for v in all_vecs
    if v.label in set(DIAG_LABELS)
    and v.method in set(DIAG_METHODS)
    and v.layer in set(DIAG_LAYERS)
    and v.extraction_spec in set(DIAG_SPECS)
]
print(f"Running diagnostics on {len(_selected_vecs)} vectors ...")

_diag_rows = []
_ll_details = {}

for _vec in _selected_vecs:
    _kws = _keywords_for(_vec.label)
    _key = f"{_vec.label} / {_vec.method} / L{_vec.layer} / {_vec.extraction_spec}"
    _row = {"label": _vec.label, "method": _vec.method, "layer": _vec.layer,
            "spec": _vec.extraction_spec, "keywords": ", ".join(_kws)}
    try:
        _readout  = ls.logit_lens_top_tokens(model, _vec, k=10) if CFG_SCORING_LOGIT_LENS else None
        _overlap  = ls.embedding_keyword_overlap(model, _vec, _kws) if CFG_SCORING_KEYWORD_OVERLAP else None
        _ablation = (ls.token_ablation_score(model, test_entries, _vec, keywords=_kws, show_progress=False)
                     if CFG_SCORING_TOKEN_ABLATION else None)
        _npca     = (ls.neutral_pca_score(model, _vec,
                         neutral_entries=random.sample(test_entries, min(20, len(test_entries))),
                         eval_entries=test_entries, n_components=4, show_progress=False)
                     if CFG_SCORING_NEUTRAL_PCA else None)
        _row.update({
            "kw_cosine":       round(_overlap.cosine, 4)                  if _overlap   else None,
            "ablation_kept":   round(_ablation.retained_fraction, 4)      if _ablation  else None,
            "neutral_pca_kept":round(_npca.retained_fraction, 4)          if _npca      else None,
            "top_up_tokens":   ", ".join(t for t, _ in _readout.top_positive[:5]) if _readout else "",
            "status": "ok",
        })
        if _readout:
            _ll_details[_key] = {"label": _vec.label, "method": _vec.method,
                                  "layer": _vec.layer, "spec": _vec.extraction_spec,
                                  "positive": [{"rank": i+1, "token": t, "logit": round(v,3)}
                                                for i,(t,v) in enumerate(_readout.top_positive)],
                                  "negative": [{"rank": i+1, "token": t, "logit": round(v,3)}
                                                for i,(t,v) in enumerate(_readout.top_negative)]}
    except Exception as _exc:
        _row.update({"kw_cosine": None, "ablation_kept": None, "neutral_pca_kept": None,
                     "top_up_tokens": "", "status": str(_exc)[:120]})
    _diag_rows.append(_row)

scoring_df = pl.DataFrame(_diag_rows)
print(f"✅ Diagnostics done — {len(scoring_df)} rows")
scoring_df.select(["label","method","layer","spec","kw_cosine","ablation_kept","neutral_pca_kept","top_up_tokens","status"])

In [ ]:
## ── DIAGNOSTIC CHARTS ────────────────────────────────────────────────────────
_pd2 = scoring_df.to_pandas()

if "kw_cosine" in _pd2.columns and _pd2["kw_cosine"].notna().any():
    display(
        alt.Chart(_pd2).mark_line(point=True).encode(
            x=alt.X("layer:O"), y=alt.Y("kw_cosine:Q", scale=alt.Scale(domain=[0,1])),
            color="method:N", tooltip=["label","method","layer","spec","kw_cosine","keywords"],
        ).properties(width=200, height=160).facet(facet=alt.Facet("label:N", columns=4),
            title="Keyword Cosine by Behaviour (lower = better, < 0.2 is good)")
    )

if "ablation_kept" in _pd2.columns and _pd2["ablation_kept"].notna().any():
    display(
        alt.Chart(_pd2).mark_bar().encode(
            x=alt.X("layer:O"), y=alt.Y("ablation_kept:Q", scale=alt.Scale(domain=[0,1])),
            color="method:N", tooltip=["label","method","layer","spec","ablation_kept"],
        ).properties(width=150, height=200).facet(facet=alt.Facet("label:N", columns=4),
            title="Token Ablation retained fraction (higher = better, > 0.7 is good)")
    )

if "neutral_pca_kept" in _pd2.columns and _pd2["neutral_pca_kept"].notna().any():
    display(
        alt.Chart(_pd2).mark_bar().encode(
            x=alt.X("layer:O"), y=alt.Y("neutral_pca_kept:Q", scale=alt.Scale(domain=[0,1])),
            color="method:N", tooltip=["label","method","layer","spec","neutral_pca_kept"],
        ).properties(width=150, height=200).facet(facet=alt.Facet("label:N", columns=4),
            title="Neutral-PCA retained fraction (higher = better, > 0.7 is good)")
    )

# Logit-lens cards (HTML)
if _ll_details:
    _cards = []
    for _dv in _ll_details.values():
        _rows_ll = _dv["positive"] + _dv["negative"]
        if not _rows_ll: continue
        _max_abs = max(abs(r["logit"]) for r in _rows_ll) or 1.0
        _bars = []
        for _side, _color in [("positive", "#a7c080"), ("negative", "#e67e80")]:
            for r in _dv[_side]:
                w = abs(r["logit"]) / _max_abs * 100
                _sign = "+" if _side == "positive" else ""
                _bars.append(
                    f'<div style="display:flex;align-items:center;gap:6px;font-size:11px;">'
                    f'<span style="width:80px;text-align:right;font-family:monospace;">{r["token"]}</span>'
                    f'<div style="flex:1;background:rgba(0,0,0,.1);height:14px;border-radius:2px;overflow:hidden;">'
                    f'<div style="width:{w:.1f}%;height:100%;background:{_color};"></div></div>'
                    f'<span style="width:50px;font-variant-numeric:tabular-nums;">{_sign}{r["logit"]:.2f}</span>'
                    f'</div>'
                )
        _cards.append(
            f'<div style="border:1px solid #475258;border-radius:6px;padding:10px;'
            f'background:#343f44;min-width:280px;flex:0 0 auto;">'
            f'<div style="font-weight:600;font-size:12px;margin-bottom:2px;">{_dv["label"]}</div>'
            f'<div style="font-size:10px;color:#9da9a0;margin-bottom:6px;">'
            f'{_dv["method"]} · L{_dv["layer"]} · {_dv["spec"]}</div>'
            + "".join(_bars) + '</div>'
        )
    from IPython.display import HTML
    display(HTML('<h4>Logit Lens (green = upweighted, red = downweighted)</h4>'
                 '<div style="display:flex;flex-wrap:wrap;gap:10px;">' + "".join(_cards) + '</div>'))

## § 7 — Entry browser

Per-token visualization of vector-based detection on a single entry.
Shows ground truth labels and cosine-similarity side by side.

In [ ]:
## ── ENTRY BROWSER ────────────────────────────────────────────────────────────
## Change these to browse different entries / vector configs:
BROWSE_ENTRY_IDX = 0                                          # index into test_entries
BROWSE_LAYER     = all_vecs.layers()[len(all_vecs.layers()) // 2]
BROWSE_SPEC      = all_vecs.specs()[0]
BROWSE_METHOD    = all_vecs.methods()[0]
BROWSE_THRESHOLD = 0.30
BROWSE_SKIP_PREFIX = True  # hide system/user prefix

import importlib as _il
import little_steer.visualization.probe_view as _pv
_il.reload(_pv)

_entry   = test_entries[BROWSE_ENTRY_IDX]
_vecs_at = all_vecs.filter(layer=BROWSE_LAYER, spec=BROWSE_SPEC, method=BROWSE_METHOD)
assert _vecs_at.vectors, "No vectors for this combination."

_det = ls.get_multilabel_token_scores(model, _entry, _vecs_at, layer=BROWSE_LAYER)
assert _det is not None, "Could not extract token scores for this entry."

_, _msg_offsets = model.format_messages_with_offsets(_entry.messages)
_role_names = {"system": "System", "user": "User", "reasoning": "Reasoning", "assistant": "Response"}
_section_markers: dict[int, str] = {}
for _msg_idx, _msg in enumerate(_entry.messages):
    _char_offset = _msg_offsets.get(_msg_idx)
    if _char_offset is None: continue
    for _tok_i, (_cs, _ce) in enumerate(_det.token_char_spans):
        if _ce > 0 and _cs >= _char_offset:
            _section_markers[_tok_i] = _role_names.get(_msg["role"], _msg["role"])
            break

_start_tok = 0
if BROWSE_SKIP_PREFIX:
    _reasoning_keys = [k for k, v in _section_markers.items() if v in ("Reasoning", "Response")]
    if _reasoning_keys:
        _start_tok = min(_reasoning_keys)

# Deduplicate labels (some vectors may share a label)
_seen_lbls: set[str] = set(); _u_labels: list[str] = []; _col_idxs: list[int] = []
for _ii, _lbl2 in enumerate(_det.labels):
    if _lbl2 not in _seen_lbls:
        _seen_lbls.add(_lbl2); _u_labels.append(_lbl2); _col_idxs.append(_ii)
_scores = _det.scores[:, _col_idxs]

# Ground truth matrix
_gt_scores = np.zeros((len(_det.tokens), len(_u_labels)), dtype=np.float32)
_lbl_to_idx = {l: i for i, l in enumerate(_u_labels)}
for _ts in _det.token_spans:
    for _lbl3 in _ts.labels:
        if _lbl3 in _lbl_to_idx:
            _gt_scores[_ts.token_start:min(_ts.token_end, len(_det.tokens)), _lbl_to_idx[_lbl3]] = 1.0

def _mk_html(scores_arr, norm=False):
    _toks   = _det.tokens[_start_tok:]
    _cspans = _det.token_char_spans[_start_tok:]
    _sc     = scores_arr[_start_tok:]
    _markers = {k - _start_tok: v for k, v in _section_markers.items() if k >= _start_tok}
    _spans  = [
        ls.TokenSpan(
            token_start=max(0, ts.token_start - _start_tok),
            token_end=ts.token_end - _start_tok, labels=ts.labels,
        )
        for ts in _det.token_spans
        if ts.token_end > _start_tok and ts.token_start - _start_tok < len(_toks)
    ]
    return _pv.render_probe_detection_html(
        tokens=_toks, token_char_spans=_cspans, scores=_sc, labels=_u_labels,
        formatted_text=_det.formatted_text, token_spans=_spans,
        threshold=BROWSE_THRESHOLD, mode="token",
        show_ground_truth=False, normalize_scores=norm,
        section_markers=_markers, show_legend=False, show_header=False,
    )

from IPython.display import HTML
_user_msg = next((m["content"] for m in _entry.messages if m["role"] == "user"), "")
print(f"Entry: {_user_msg[:100]}{'…' if len(_user_msg) > 100 else ''}")
print(f"Layer={BROWSE_LAYER} · Spec={BROWSE_SPEC} · Method={BROWSE_METHOD}")
display(HTML(_pv.legend_html(_u_labels, BROWSE_THRESHOLD)))
display(HTML("<h4>Ground truth</h4>" + _mk_html(_gt_scores, norm=False)))
display(HTML("<h4>Vector detection</h4>" + _mk_html(_scores, norm=True)))

## § 8 — Steering playground

Inject a chosen steering vector during generation and compare to the baseline.
**Alpha sweep** generates at multiple strengths so you can see the effect curve.

In [ ]:
## ── STEERING: BASELINE vs STEERED ───────────────────────────────────────────
## Change these to pick a different vector / prompt:
STEER_LABEL  = CFG_STEER_LABEL  or all_vecs.labels()[0]
STEER_METHOD = CFG_STEER_METHOD or all_vecs.methods()[0]
STEER_LAYER  = CFG_STEER_LAYER  or all_vecs.layers()[len(all_vecs.layers()) // 2]
STEER_SPEC   = CFG_STEER_SPEC   or all_vecs.specs()[0]
STEER_ALPHA  = CFG_STEER_ALPHA
STEER_PROMPT = CFG_STEER_PROMPT
STEER_MAX_TOKENS = CFG_STEER_MAX_TOKENS

_vs = all_vecs.filter(label=STEER_LABEL, method=STEER_METHOD, layer=STEER_LAYER, spec=STEER_SPEC)
assert _vs.vectors, f"No steering vector for label={STEER_LABEL!r} method={STEER_METHOD!r} layer={STEER_LAYER} spec={STEER_SPEC!r}"
_raw_vec = _vs.vectors[0].vector.float()

_msgs = [{"role": "user", "content": STEER_PROMPT}]
print(f"Generating baseline ...")
_baseline = ls.steered_generate(model, _msgs, max_new_tokens=STEER_MAX_TOKENS, do_sample=False)

print(f"Generating steered (α={STEER_ALPHA}) ...")
_steered = ls.steered_generate(
    model, _msgs, steering_vec=_raw_vec, layer=STEER_LAYER,
    alpha=STEER_ALPHA, response_only=True,
    max_new_tokens=STEER_MAX_TOKENS, do_sample=False,
)

from IPython.display import HTML
display(HTML(
    f"<h4>Prompt</h4><blockquote>{STEER_PROMPT}</blockquote>"
    f"<p><b>Steering:</b> label={STEER_LABEL!r} · method={STEER_METHOD!r} · layer={STEER_LAYER} · spec={STEER_SPEC!r} · α={STEER_ALPHA}</p>"
    f"<table style='width:100%;border-collapse:collapse'>"
    f"<tr><th style='width:50%;padding:8px;background:#343f44;'>Baseline</th>"
    f"<th style='padding:8px;background:#3d484d;'>Steered</th></tr>"
    f"<tr><td style='padding:12px;vertical-align:top;'>{_baseline}</td>"
    f"<td style='padding:12px;vertical-align:top;background:#3d484d;'>{_steered}</td></tr>"
    f"</table>"
))

In [ ]:
## ── ALPHA SWEEP ──────────────────────────────────────────────────────────────
## Generates at 5 alpha values from 0 → STEER_ALPHA. Uses the same vector as
## the cell above. Re-run just this cell after changing STEER_ALPHA etc.
SWEEP_N_STEPS = 5  # number of alpha steps

_alphas = [round(float(a), 2) for a in np.linspace(0.0, STEER_ALPHA, SWEEP_N_STEPS)]
_sweep_outputs = {}
for _a in _alphas:
    print(f"  α={_a:.1f} ...", end="\r")
    _sweep_outputs[_a] = ls.steered_generate(
        model, _msgs,
        steering_vec=_raw_vec if _a > 0 else None,
        layer=STEER_LAYER if _a > 0 else None,
        alpha=_a, response_only=True,
        max_new_tokens=STEER_MAX_TOKENS, do_sample=False,
    )

print(f"\n✅ Alpha sweep done")
_cells = []
for _a in _alphas:
    _bg = "#232a2e" if _a == 0 else "#3d484d"
    _cells.append(
        f'<td style="vertical-align:top;padding:10px;background:{_bg};width:{100//SWEEP_N_STEPS}%;">'
        f'<b>α = {_a}</b><br><br>{_sweep_outputs[_a]}</td>'
    )
display(HTML(
    f"<h4>Alpha sweep · {STEER_LABEL!r} · {STEER_METHOD!r} · L{STEER_LAYER} · {STEER_SPEC!r}</h4>"
    f"<table style='width:100%;border-collapse:collapse;'><tr>{''.join(_cells)}</tr></table>"
))

In [ ]:
## ── LAYER COMPARISON ─────────────────────────────────────────────────────────
## Steer at several layers simultaneously and compare outputs side-by-side.
## Change COMPARE_LAYERS to any subset of all_vecs.layers():
COMPARE_LAYERS = all_vecs.layers()[::max(1, len(all_vecs.layers()) // 5)]  # ~5 evenly spaced layers

_layer_outputs = {}
for _cl in COMPARE_LAYERS:
    print(f"  Steering at layer {_cl} ...", end="\r")
    _vs_cl = all_vecs.filter(label=STEER_LABEL, method=STEER_METHOD, layer=_cl, spec=STEER_SPEC)
    if not _vs_cl.vectors:
        _layer_outputs[_cl] = f"_(no vector at layer {_cl})_"
        continue
    _r_cl = _vs_cl.vectors[0].vector.float()
    _layer_outputs[_cl] = ls.steered_generate(
        model, _msgs, steering_vec=_r_cl, layer=_cl,
        alpha=STEER_ALPHA, response_only=True,
        max_new_tokens=STEER_MAX_TOKENS, do_sample=False,
    )

print(f"\n✅ Layer comparison done")
_cols = [f'<td style="vertical-align:top;padding:10px;background:#343f44;width:{100//(len(COMPARE_LAYERS)+1)}%;">'
         f'<b>Baseline</b><br><br>{_baseline}</td>']
for _cl in COMPARE_LAYERS:
    _cols.append(
        f'<td style="vertical-align:top;padding:10px;background:#3d484d;width:{100//(len(COMPARE_LAYERS)+1)}%;">'
        f'<b>Layer {_cl}</b><br><br>{_layer_outputs[_cl]}</td>'
    )
display(HTML(
    f"<h4>Layer comparison · {STEER_LABEL!r} · {STEER_METHOD!r} · α={STEER_ALPHA}</h4>"
    f"<table style='width:100%;border-collapse:collapse;'><tr>{''.join(_cols)}</tr></table>"
))